# Egyptian National ID Card — Data Extraction Pipeline

Pipeline: **Crop card → Detect fields → Preprocess → OCR → DataFrame**

Two YOLOv8 models:
- **Crop model**: detects card boundary (`crn` / `nid`) and crops it
- **Fields model**: detects 15 fields (front + back combined)

OCR: PaddleOCR (Arabic)

## 1. Setup & Imports

In [ ]:
# Check GPU availability
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Install YOLO
!pip install ultralytics -q

In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

## 2. Data Preparation

Copy datasets to `/kaggle/working` (writable) and fix `data.yaml` paths to absolute paths.

In [ ]:
# Copy datasets safely (overwrite if already exists — /kaggle/working persists across restarts)
def safe_copytree(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Copied: {dst}")

safe_copytree('/kaggle/input/datasets/stud20230837/all-fields-dataset', '/kaggle/working/all-fields-dataset')
safe_copytree('/kaggle/input/datasets/stud20230837/crop-data/croping_data', '/kaggle/working/crop-data')

In [ ]:
# Fix data.yaml files with absolute paths

yaml_content_crop = """train: /kaggle/working/crop-data/train/images
val: /kaggle/working/crop-data/valid/images
test: /kaggle/working/crop-data/test/images

nc: 2
names: ['crn', 'nid']
"""
with open('/kaggle/working/crop-data/data.yaml', 'w') as f:
    f.write(yaml_content_crop)

yaml_content_fields = """train: /kaggle/working/all-fields-dataset/train/images
val: /kaggle/working/all-fields-dataset/valid/images
test: /kaggle/working/all-fields-dataset/test/images

nc: 15
names: ['address', 'birth_date', 'country', 'doc_type', 'education', 'expire_date', 'gender',
        'husband', 'image', 'issue_date', 'job', 'martial_state', 'name', 'national_id', 'religion']
"""
with open('/kaggle/working/all-fields-dataset/data.yaml', 'w') as f:
    f.write(yaml_content_fields)

print("data.yaml files updated")

In [ ]:
# Sanity check: confirm images exist
for path in [
    '/kaggle/working/all-fields-dataset/train/images',
    '/kaggle/working/all-fields-dataset/valid/images',
    '/kaggle/working/crop-data/train/images',
    '/kaggle/working/crop-data/valid/images',
]:
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"{path} -> {count} images")

## 3. Model 1 — Crop Model

Detects the card boundary (`crn` or `nid`) so it can be cropped from the background.

In [ ]:
model_crop = YOLO('yolov8n.pt')

results_crop = model_crop.train(
    data='/kaggle/working/crop-data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='crop_model'
)

In [ ]:
# Load best weights after training
model_crop = YOLO('/kaggle/working/runs/detect/crop_model/weights/best.pt')

In [ ]:
def crop_card(image_path, model, conf_threshold=0.25, padding=30):
    """
    Detects the card in an image and returns the cropped card + card type (crn/nid).
    Returns (None, None) if no card is detected.
    """
    results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

    if len(results[0].boxes) == 0:
        print("No card detected")
        return None, None

    boxes = results[0].boxes
    best_box = boxes[boxes.conf.argmax().item()]

    class_id = int(best_box.cls[0])
    card_type = model.names[class_id]
    x1, y1, x2, y2 = best_box.xyxy[0].tolist()

    original_image = cv2.imread(image_path)
    img_h, img_w = original_image.shape[:2]

    x1 = max(0, int(x1) - padding)
    y1 = max(0, int(y1) - padding)
    x2 = min(img_w, int(x2) + padding)
    y2 = min(img_h, int(y2) + padding)

    cropped_image = original_image[y1:y2, x1:x2]
    return cropped_image, card_type

### Quick test — Crop Model

In [ ]:
test_images_path = '/kaggle/working/crop-data/test/images'
test_images = os.listdir(test_images_path)
test_image_path = os.path.join(test_images_path, test_images[0])

cropped_img, card_type = crop_card(test_image_path, model_crop)

if cropped_img is not None:
    print(f"Card type: {card_type} | shape: {cropped_img.shape}")
    plt.figure(figsize=(6, 6))
    plt.imshow(cropped_img[..., ::-1])
    plt.axis('off')
    plt.title(f"Cropped Card - {card_type}")
    plt.show()

## 4. Model 2 — Fields Model

Detects the 15 card fields (front + back combined) inside the cropped card.

In [ ]:
model_fields = YOLO('yolov8n.pt')

results_fields = model_fields.train(
    data='/kaggle/working/all-fields-dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='fields_model'
)

In [ ]:
# Load best weights after training
model_fields = YOLO('/kaggle/working/runs/detect/fields_model/weights/best.pt')

In [ ]:
# Per-field padding (some fields need more margin than others)
FIELD_PADDING = {
    "address": 15, "name": 12, "job": 10, "religion": 5,
    "martial_state": 10, "husband": 1, "education": 10,
    "national_id": 5, "birth_date": 8, "issue_date": 8,
    "expire_date": 8, "gender": 5
}

def detect_fields(cropped_card_image, model, conf_threshold=0.25, iou_threshold=0.5):
    """
    Detects all fields inside a cropped card.
    Returns a dict: {field_name: {"image": crop, "confidence": float, "bbox": [...]}}
    Keeps only the highest-confidence box per field.
    """
    results = model.predict(source=cropped_card_image, conf=conf_threshold, iou=iou_threshold, verbose=False)

    fields_dict = {}
    if len(results[0].boxes) == 0:
        print("No fields detected")
        return fields_dict

    img_h, img_w = cropped_card_image.shape[:2]

    for box in results[0].boxes:
        class_id = int(box.cls[0])
        field_name = model.names[class_id]
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        padding = FIELD_PADDING.get(field_name, 1)
        x1 = max(0, int(x1) - padding)
        y1 = max(0, int(y1) - padding)
        x2 = min(img_w, int(x2) + padding)
        y2 = min(img_h, int(y2) + padding)

        field_crop = cropped_card_image[y1:y2, x1:x2]

        if field_name not in fields_dict or confidence > fields_dict[field_name]["confidence"]:
            fields_dict[field_name] = {"image": field_crop, "confidence": confidence, "bbox": [x1, y1, x2, y2]}

    return fields_dict

### Quick test — Fields Model

In [ ]:
fields = detect_fields(cropped_img, model_fields)

for field_name, data in fields.items():
    print(f"{field_name:15s} | conf: {data['confidence']:.3f}")

In [ ]:
# Visualize each detected field
for field_name, data in fields.items():
    plt.figure(figsize=(4, 2))
    plt.imshow(data['image'][..., ::-1])
    plt.title(f"{field_name} (conf: {data['confidence']:.2f})")
    plt.axis('off')
    plt.show()

## 5. OCR Setup — PaddleOCR

Note: pin `paddlepaddle` version to avoid API compatibility errors with `paddleocr`.

In [ ]:
!pip install paddlepaddle==3.2.2 -q
!pip install paddleocr -q

In [ ]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='ar')

## 6. Preprocessing & Text Extraction

- `CONSTANT_FIELDS`: fixed text printed on every card (skip OCR)
- `preprocess_field()`: field-specific image cleanup before OCR
- `extract_text_ordered()`: runs OCR and orders text top-to-bottom, right-to-left (correct reading order for Arabic)

In [ ]:
CONSTANT_FIELDS = {
    "country": "جمهورية مصر العربية",
    "doc_type": "بطاقة تحقيق الشخصية"
}

In [ ]:
def preprocess_field(img, class_name):
    """Field-specific preprocessing before OCR."""
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if class_name == "image":
        return img

    if class_name == "national_id":
        img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = clahe.apply(img)

    elif class_name in ["birth_date", "issue_date", "expire_date"]:
        img = cv2.resize(img, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
        img = cv2.fastNlMeansDenoising(img, None, 7, 7, 20)

    elif class_name == "name":
        img = cv2.resize(img, None, fx=2, fy=1.5, interpolation=cv2.INTER_CUBIC)
        img = cv2.fastNlMeansDenoising(img, None, 11, 11, 20)

    elif class_name == "address":
        img = cv2.fastNlMeansDenoising(img, None, 15, 3, 5)
        img = cv2.copyMakeBorder(img, 41, 41, 41, 41, cv2.BORDER_CONSTANT, value=255)
        img = cv2.resize(img, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)

    elif class_name in ["gender", "religion", "martial_state", "education", "job", "husband"]:
        img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        img = cv2.fastNlMeansDenoising(img, None, 11, 7, 20)

    else:
        img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = clahe.apply(img)

    return img

In [ ]:
def extract_text_ordered(field_image, ocr_model, line_threshold=15):
    """
    Runs PaddleOCR and orders detected text lines top-to-bottom,
    then right-to-left within each line (correct for Arabic).
    """
    if len(field_image.shape) == 2:
        field_image = cv2.cvtColor(field_image, cv2.COLOR_GRAY2BGR)

    result = ocr_model.predict(field_image, use_doc_orientation_classify=False, use_doc_unwarping=False)

    if len(result) == 0 or 'rec_texts' not in result[0]:
        return []

    texts = result[0]['rec_texts']
    boxes = result[0]['rec_boxes']
    if len(texts) == 0:
        return []

    items = []
    for text, box in zip(texts, boxes):
        y_center = (box[1] + box[3]) / 2
        x_right = box[0]
        items.append((y_center, x_right, text))

    # sort by line (rounded Y), then right-to-left (X) within each line
    items.sort(key=lambda item: (round(item[0] / line_threshold), -item[1]))

    return [item[2] for item in items]

## 7. Test — Full Pipeline on One Card

In [ ]:
processed_fields = {}
final_results = {}

for field_name, data in fields.items():
    if field_name in CONSTANT_FIELDS:
        final_results[field_name] = CONSTANT_FIELDS[field_name]
        continue

    processed_img = preprocess_field(data['image'], field_name)
    processed_fields[field_name] = processed_img

    if field_name == "image":
        final_results[field_name] = "(image)"
        continue

    texts = extract_text_ordered(processed_img, ocr)
    final_results[field_name] = " ".join(texts) if texts else "not read"

print("=== Result ===")
for field_name, text in final_results.items():
    print(f"{field_name:15s} -> {text}")

In [ ]:
# Visualize processed fields
n_fields = len(processed_fields)
fig, axes = plt.subplots(n_fields, 1, figsize=(8, n_fields * 2))
if n_fields == 1:
    axes = [axes]
for ax, (field_name, img) in zip(axes, processed_fields.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(field_name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Batch Processing — Folder of Images to DataFrames

Processes a folder of mixed front/back card images and splits results into
two DataFrames (front fields vs. back fields), since a single image only
contains one side of the card.

In [ ]:
FRONT_ONLY_FIELDS = {'name', 'address', 'birth_date'}
BACK_ONLY_FIELDS = {'gender', 'religion', 'martial_state', 'job', 'education', 'husband', 'issue_date', 'expire_date'}

FRONT_COLUMNS = ['source_image', 'name', 'national_id', 'address', 'birth_date', 'country', 'doc_type']
BACK_COLUMNS = ['source_image', 'national_id', 'gender', 'religion', 'martial_state', 'job', 'education', 'husband', 'issue_date', 'expire_date']


def process_single_image(image_path, model_crop, model_fields, ocr_model):
    """Runs the full pipeline (crop -> detect -> preprocess -> OCR) on one image."""
    cropped_img, card_type = crop_card(image_path, model_crop)
    if cropped_img is None:
        return None, None

    fields = detect_fields(cropped_img, model_fields)
    if len(fields) == 0:
        return None, None

    row = {"source_image": os.path.basename(image_path)}

    for field_name, data in fields.items():
        if field_name in CONSTANT_FIELDS:
            row[field_name] = CONSTANT_FIELDS[field_name]
            continue
        if field_name == "image":
            row[field_name] = "(image)"
            continue

        processed_img = preprocess_field(data['image'], field_name)
        texts = extract_text_ordered(processed_img, ocr_model)
        row[field_name] = " ".join(texts) if texts else "not read"

    return row, set(fields.keys())


def build_dataframes_from_folder(folder_path, model_crop, model_fields, ocr_model):
    """Processes every image in a folder, splitting results into front/back DataFrames."""
    front_rows, back_rows, skipped_files = [], [], []

    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(image_files)} images")

    for filename in image_files:
        image_path = os.path.join(folder_path, filename)
        row, detected_fields = process_single_image(image_path, model_crop, model_fields, ocr_model)

        if row is None:
            skipped_files.append(filename)
            print(f"{filename}: skipped (no card/fields detected)")
            continue

        is_front = len(detected_fields & FRONT_ONLY_FIELDS) > 0
        is_back = len(detected_fields & BACK_ONLY_FIELDS) > 0

        if is_front:
            front_rows.append({col: row.get(col, "-") for col in FRONT_COLUMNS})
            print(f"{filename}: front")
        elif is_back:
            back_rows.append({col: row.get(col, "-") for col in BACK_COLUMNS})
            print(f"{filename}: back")
        else:
            skipped_files.append(filename)
            print(f"{filename}: could not classify")

    df_front = pd.DataFrame(front_rows)
    df_back = pd.DataFrame(back_rows)

    print(f"\nSummary: front={len(df_front)}, back={len(df_back)}, skipped={len(skipped_files)}")
    return df_front, df_back

In [ ]:
folder_path = "/kaggle/input/datasets/stud20230837/test-imge/test_images"

df_front, df_back = build_dataframes_from_folder(folder_path, model_crop, model_fields, ocr)

print("=== Front face ===")
display(df_front)

print("\n=== Back face ===")
display(df_back)

## 9. Quick Reload (run this after any Kaggle session restart)

Trained models are saved on disk and survive a restart — only variables in memory are lost.
Run this single cell instead of re-running the whole notebook.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from paddleocr import PaddleOCR

model_crop = YOLO('/kaggle/working/runs/detect/crop_model/weights/best.pt')
model_fields = YOLO('/kaggle/working/runs/detect/fields_model/weights/best.pt')
ocr = PaddleOCR(use_angle_cls=True, lang='ar')

print("Models and OCR reloaded")